# Universal Documentary Studio — Colab GPU Worker

This notebook is the primary GPU worker (see `SETUP_COLAB.md`). It:
1. installs dependencies
2. detects GPU / VRAM / CUDA (never assumes a fixed GPU)
3. initializes `ResourceManager` and `ModelRegistry`
4. optionally mounts Google Drive for persistent project storage
5. runs the pipeline (or a worker loop) with automatic checkpointing/resume

In [ ]:
# 1. Clone / upload the repository, then install dependencies.
# If running from a zip upload instead of git, skip the clone line.
# !git clone <your-repo-url> uds
%cd /content/uds
!pip install -q -r requirements.txt

In [ ]:
# 2 + 3. Detect hardware and initialize the resource-aware components.
# This NEVER downloads a heavyweight model automatically (spec section 59).
import sys
sys.path.insert(0, '/content/uds')

from app.startup import run_startup
summary = run_startup(lightweight=True)
summary

In [ ]:
# 4. (Optional) Mount Google Drive for persistent project storage across
# sessions. Recommended: Colab sessions can terminate without warning.
MOUNT_DRIVE = False  # set True to enable
PROJECTS_ROOT = 'projects'

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECTS_ROOT = '/content/drive/MyDrive/uds_projects'
    import os
    os.makedirs(PROJECTS_ROOT, exist_ok=True)

print('Using projects root:', PROJECTS_ROOT)

In [ ]:
# 5. Run (or resume) a project. Re-running this cell for the SAME topic
# after a disconnect will resume from the last completed checkpoint
# rather than starting over -- see core/project_manager.py.
from core.models import ProjectConfig
from core.project_manager import ProjectManager
from core.pipeline import DocumentaryPipeline

TOPIC = "How a Small Technology Company Became a Global Leader"

config = ProjectConfig(
    topic=TOPIC,
    target_duration_minutes=10.0,
    short_count=4,
    mock_mode=True,       # flip to False once real models are wired in
    local_gpu_enabled=False,  # this notebook IS the remote worker; keep
                              # the local-machine safeguard semantics intact
)

pm = ProjectManager(projects_root=PROJECTS_ROOT, config=config)
print('Project ID:', config.project_id)
print('Resuming from state:', pm.state_machine.current_state.value)
print('Completed checkpoints:', pm.resume_summary()['completed_checkpoints'])

pipeline = DocumentaryPipeline(pm, video_width=1920, video_height=1080, fps=24)
result = pipeline.run_full()

print('Final state:', pm.state_machine.current_state.value)
print('QA score:', result.qa_report.score, result.qa_report.status.value)
print('Long-form render:', result.render_path)
print('Shorts:', result.short_render_paths)

## Notes

- If this session disconnects mid-run, just reopen the notebook,
  re-run cells 1-4, set the same `TOPIC` (and `PROJECTS_ROOT` if you
  mounted Drive), and re-run cell 5. It will pick up from the last
  completed checkpoint automatically.
- `ResourceManager` reserves a safety margin below detected VRAM before
  considering any job schedulable; it will never attempt to use 100% of
  available VRAM.
- To wire in real (non-mock) models, register additional
  `ModelCapability` entries via `ModelRegistry.register()` and swap the
  relevant adapter's mock implementation for a real one — no other code
  in the pipeline needs to change, since every agent depends only on the
  adapter interfaces in `adapters/*/base.py`.